In [1]:
import torch
\
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Torch version: 2.6.0+cu124
CUDA available: True
GPU Name: NVIDIA GeForce RTX 4070 Ti


In [2]:
import torchtune

/home/admingwi/anaconda3/envs/torch_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import rdkit
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, MACCSkeys

In [4]:
import torch
from transformers import AutoModel, AutoTokenizer

# Название модели на Hugging Face
# Варианты: aaronfeller/PeptideMTR_xs, aaronfeller/PeptideMTR_sm, aaronfeller/PeptideMTR_lg и т.д.
MODEL_NAME = "aaronfeller/PeptideMTR_lg"

print(f"Загрузка модели {MODEL_NAME}...")

# Загружаем токенизатор и модель.
# Важно: trust_remote_code=True необходим, так как автор использует кастомную архитектуру/токенизацию.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Переводим модель в режим оценки (eval), чтобы отключить dropout и т.д.
model.eval()

Загрузка модели aaronfeller/PeptideMTR_lg...


Loading weights: 100%|██████████| 325/325 [00:00<00:00, 833.63it/s, Materializing param=model.transformer.norm.weight]                   


MLM_model(
  (model): MLM_core(
    (embed): Embedding(405, 1024)
    (transformer): TransformerStack(
      (blocks): ModuleList(
        (0-31): 32 x UnifiedTransformerBlock(
          (attn_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (attn): MultiHeadAttention(
            (qkv_proj): Linear(in_features=1024, out_features=3072, bias=False)
            (rotary): RotaryPositionalEmbeddings()
            (out_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (ffn_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (ffn): SwiGLU(
            (linear1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear2): Linear(in_features=2048, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    )
    (sequence_h

In [5]:
path = 'Data'
folder = 'Caco'
data_name = 'caco_b'
set_name = 'train'
func_y = 'Caco-2_prm'

In [6]:
path = 'Data'
folder = 'Cell_p'
data_name = 'Cell_p'
set_name = 'test'
func_y = 'CellP'

In [25]:
path = 'Data'
folder = 'Half_life'
data_name = 'Half_life'
set_name = 'train_organ'
#func_y = 'Half-life, h (Homo sapiens)'
func_y = 'Half-life, h'

In [8]:
path = 'Data'
folder = 'Toxicity'
data_name = 'Hemotox'
set_name = 'train'
func_y = 'Tox'

In [9]:
hl_df = pd.read_excel(f'{path}/{folder}/{data_name}_{set_name}.xlsx')
hl_df

,Sequence,Tox,Source,Smiles
0,AAAAAAAAAAGIGKFLHSAKKFGKAFVGEIMNS,0,HemTox,CC[C@H](C)[C@H](NC(=O)CNC(=O)[C@H](C)NC(=O)[C@...
1,AAAKAALNAVLVGANA,1,"Toxipep, HemTox",CC(C)C[C@H](NC(=O)[C@H](C)NC(=O)[C@H](C)NC(=O)...
2,AAGKGLVSNLLEK,1,HemTox,CC(C)C[C@H](NC(=O)CNC(=O)[C@H](CCCCN)NC(=O)CNC...
3,AAGLAMLFLGILSAAGSTMGARA,1,HemTox,CC[C@H](C)[C@H](NC(=O)CNC(=O)[C@H](CC(C)C)NC(=...
4,AAGMGFFGAR,1,"ToxinPred3, Toxipep, HemTox",CSCC[C@H](NC(=O)CNC(=O)[C@H](C)NC(=O)[C@H](C)N...
...,...,...,...,...
5698,SLWENFKNAGKK,0,HemoPI2,CC(C)C[C@H](NC(=O)[C@@H](N)CO)C(=O)N[C@@H](Cc1...
5699,ALWKTMLKKLGTMALHAGK,0,HemoPI2,CSCC[C@H](NC(=O)[C@@H](NC(=O)CNC(=O)[C@H](CC(C...
5700,KIAGKIAKIAGKIAKIAGKIA,1,HemoPI2,CC[C@H](C)[C@H](NC(=O)[C@H](CCCCN)NC(=O)CNC(=O...
5701,AKVVKKLTKGVAKLLK,0,HemoPI2,CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CCC...


In [27]:
categories = ['Homo sapiens', 'Rattus norvegicu', 'Mus musculus', 'Macaca fascicularis', 'Canis lupus']  # Порядок важен!
mapping = {cat: i for i, cat in enumerate(categories)}
hl_df['Org_ind'] = hl_df['Organism'].map(mapping)
hl_df

,sources,Name,Canonical_smiles,Cyclic/Linear (checked),Functional_activity,PK_groups,Canonical/Non-canonical,Sequence,Set,Organism,"Half-life, h",is_Canis lupus,is_Homo sapiens,is_Macaca fascicularis,is_Mus musculus,is_Rattus norvegicus,Org_ind
0,"Peplife2, drugbank, thpdb",Corticotropin,CSCC[C@H](NC(=O)[C@H](CO)NC(=O)[C@H](Cc1ccc(O)...,Linear,Гормон,Группа 2: Системные метаболические регуляторы,Canonical,SYSMEHFRWGKPVGKKRRPVKVYPNGAEDESAEAFPLEF,Train_Global,Homo sapiens,0.250000,False,True,False,False,False,0.0
1,"PepMSND, Peplife2, drugbank, thpdb",Pramlintide,CC[C@H](C)[C@H](NC(=O)[C@@H]1CCCN1C(=O)CNC(=O)...,Cyclic,Противодиабетическое,Группа 2: Системные метаболические регуляторы,Canonical,KCNTATCATQRLANFLVHSSNNFGPILPPTNVGSNTY-NH2,Train_Global,Homo sapiens,0.800000,False,True,False,False,False,0.0
2,drugbank,Lipegfilgrastim,CC[C@H](C)[C@H](NC(=O)CNC(=O)[C@H](CCC(=O)O)NC...,Linear,Противоопухолевое,Группа 4: Рецепторно-сигнальные и таргетные мо...,Canonical,MTPLGPASSLPQSFLLKCLEQVRKIQGDGAALQEKLCATYKLCHPE...,Train_Global,Homo sapiens,47.000000,False,True,False,False,False,0.0
3,drugbank,Pasireotide,NCCCC[C@@H]1NC(=O)[C@@H](Cc2c[nH]c3ccccc23)NC(...,Cyclic,Гормон,Группа 2: Системные метаболические регуляторы,"Non-canonical (неканонические ак, D-ак)",Hyp-dPhg-wK-Y(Bn)-F,Train_Global,Homo sapiens,12.000000,False,True,False,False,False,0.0
4,drugbank,Peginterferon beta-1a,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](Cc1c...,Linear,Иммуномодулятор,Группа 3: Макромолекулярные и регенеративные ф...,Canonical,MSYNLLGFLQRSSNFQCQKLLWQLNGRLEYCLKDRMNFDIPEEIKQ...,Train_Global,Homo sapiens,78.000000,False,True,False,False,False,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
884,Peplife2,SANSNPAMAPRERKAGCKNFFWKTFTSC,CSCC[C@H](NC(=O)[C@H](C)NC(=O)[C@@H]1CCCN1C(=O...,NaN,NaN,NaN,NaN,SANSNPAMAPRERKAGCKNFFWKTFTSC,Train_Global,Canis lupus,0.046667,True,False,False,False,False,4.0
885,Peplife2,SPKMMHKSGCFGRRLDRIGSLSGLGCNVLRKY,CC[C@H](C)[C@H](NC(=O)[C@H](CCCNC(=N)N)NC(=O)[...,NaN,NaN,NaN,NaN,SPKMMHKSGCFGRRLDRIGSLSGLGCNVLRKY,Train_Global,Canis lupus,0.026167,True,False,False,False,False,4.0
886,Peplife2,TVRTSAD,CC(C)[C@H](NC(=O)[C@@H](N)[C@@H](C)O)C(=O)N[C@...,NaN,NaN,NaN,NaN,TVRTSAD,Train_Global,Canis lupus,1.789333,True,False,False,False,False,4.0
887,Peplife2,WAGGDASGE,C[C@H](NC(=O)[C@H](CC(=O)O)NC(=O)CNC(=O)CNC(=O...,NaN,NaN,NaN,NaN,WAGGDASGE,Train_Global,Canis lupus,0.120667,True,False,False,False,False,4.0


In [21]:
hl_df['Org_ind']=0.0

In [10]:
from tqdm import tqdm
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM
#from .autonotebook import tqdm as notebook_tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



BATCH_SIZE = 64 # Настройте под объем памяти вашей видеокарты (16, 32 или 64)

# Список всех SMILES
smiles_list = hl_df['Smiles'].tolist()

# Создаем DataLoader для автоматического разбиения на пакеты
dataloader = DataLoader(smiles_list, batch_size=BATCH_SIZE, shuffle=False)

x = []

print(f"Начинаю обработку {len(smiles_list)} структур на {device}...")

model.to(device)

all_embeddings = []

model.eval() # Перевод в режим инференса
for batch in tqdm(dataloader, desc="Extracting ProtMTR embeddings"):
    # Токенизация пакета
    # Для пептидов/SMILES в MTR моделях важно убедиться, что tokenizer 
    # настроен именно под их специфический словарь
    inputs = tokenizer(
        batch, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=512
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        
        # Извлекаем эмбеддинг [CLS] токена (индекс 0)
        # Это репрезентация всей структуры в моделях MTR
        batch_embeddings = outputs['mean_pool']
            
    # Переносим на CPU и сразу конвертируем
    all_embeddings.append(batch_embeddings.cpu().numpy())
    
    # Опционально: очистка кэша GPU для очень больших батчей
    # torch.cuda.empty_cache()

# Объединяем список массивов в один финальный массив (N_samples, Hidden_Dim)
x_df = pd.DataFrame(np.vstack(all_embeddings))

# Если вам нужно сохранить и целевую переменную y, сохраняя порядок:
#y = hl_df['Half-life, h'].values

print(f"Готово! Размер матрицы признаков: {x_df.shape}")
x_df['Canonical_smiles'] = hl_df['Smiles']
#x_df['y'] = np.log10(hl_df[func_y])
#x_df['Org_ind'] = hl_df['Org_ind']
x_df['y'] = hl_df[func_y]
x_df.to_csv(f'{path}/{folder}/{data_name}_{set_name}_pmtr.csv', index=None)

Using device: cuda
Начинаю обработку 5703 структур на cuda...


Extracting ProtMTR embeddings: 100%|██████████| 90/90 [01:07<00:00,  1.34it/s]


Готово! Размер матрицы признаков: (5703, 1024)


In [24]:
x_df

,0,1,2,3,4,5,6,7,8,9,...,1017,1018,1019,1020,1021,1022,1023,Canonical_smiles,y,Org_ind
0,-0.351591,0.651641,0.407053,-0.524015,0.051456,-0.175234,-0.164901,0.369037,0.256546,-0.405084,...,0.446379,0.029607,0.032372,-0.116973,-0.133006,-0.065409,0.171808,CSCC[C@H](NC(=O)[C@H](CO)NC(=O)[C@H](Cc1ccc(O)...,-0.602060,0.0
1,-0.332118,0.615787,0.391367,-0.505294,0.049221,-0.173746,-0.119629,0.318082,0.261264,-0.311955,...,0.467819,0.051519,0.022395,-0.023385,-0.056153,-0.051592,0.198477,CC[C@H](C)[C@H](NC(=O)[C@@H]1CCCN1C(=O)CNC(=O)...,-0.096910,0.0
2,0.274847,-0.465732,0.466423,-0.060096,0.799861,-1.296159,0.473422,0.613404,1.411271,-0.112265,...,0.204110,-0.044140,0.470568,-0.318533,-0.099717,0.837568,0.830676,CC[C@H](C)[C@H](NC(=O)CNC(=O)[C@H](CCC(=O)O)NC...,1.672098,0.0
3,-0.140931,0.291383,0.130711,-0.228193,0.004982,-0.011204,0.014940,0.091686,0.070850,-0.090486,...,0.103748,0.006484,0.035158,-0.002948,0.012361,-0.067794,0.020682,NCCCC[C@@H]1NC(=O)[C@@H](Cc2c[nH]c3ccccc23)NC(...,1.079181,0.0
4,-1.097836,1.842018,0.985411,-1.451292,-0.014585,-0.440717,-0.136246,0.822383,0.791078,-0.726246,...,1.181469,0.187424,0.018052,-0.142716,-0.150380,-0.063276,0.470184,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](Cc1c...,1.892095,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,-0.969194,1.783713,1.016028,-1.291628,-0.006519,-0.374745,-0.301519,0.861378,0.725211,-0.954075,...,0.961665,0.062024,0.105334,-0.317312,-0.253943,-0.257492,0.310230,CC[C@H](C)[C@H](NC(=O)[C@H](CO)NC(=O)[C@H](Cc1...,0.778151,0.0
287,0.105028,0.013117,0.095882,0.187470,0.244625,-0.320210,0.178518,0.043595,-0.001909,-0.138423,...,0.107730,-0.031973,-0.045228,-0.034233,0.250848,0.180941,0.067653,N=C(N)NCCC[C@H](NC(=O)[C@H](CCCNC(=N)N)NC(=O)[...,0.824126,0.0
288,-0.127928,0.160877,0.040546,-0.209925,0.046523,-0.000541,-0.065947,0.115456,-0.035162,-0.086738,...,0.115612,-0.028314,0.027932,-0.030489,-0.034804,-0.087194,0.094417,NC(=O)[C@@H]1CCCN1C(=O)[C@H](Cc1ccccc1)NC(=O)[...,-0.356547,0.0
289,-0.243833,0.492816,0.277889,-0.383546,0.003443,-0.101130,-0.164390,0.259333,0.137350,-0.406202,...,0.271872,-0.062278,0.008166,-0.156597,-0.096592,-0.185023,0.100709,CSCC[C@H](NC(=O)[C@@H](NC(=O)[C@@H](N)Cc1ccc(O...,-1.522879,0.0
